# How long will it take for larger and larger graphs?

How does absorption time (and therefore compute cost) scale with population size `N`?
Linear, polynomial, or exponential? And what would a batch at N = 2000 or N = 10000 cost?

## Design

**Ladder.** 13 log-spaced sizes, 10 to 1000. Log spacing because the fit consumes lever arm in
`log N`: linear spacing (10, 20, ... 1000) puts 90% of its points in the last half-decade, which
is also where each point is most expensive.

**Topology.** `E = round(1.1 * N)`, i.e. mean degree `k = 2.2` held constant. This matches the real
`avian_r4_l7` graph (N=31, E=34, k=2.19) at *every* rung. Constant *density* would not: density
0.073 means k=2.2 at N=31 but k=73 at N=1000, so a fit through it would mix N with connectivity,
and the zoo would be 1.5 GB instead of ~50 MB.

## Two different "times" - read this before interpreting anything

The word "time" in the title covers two quantities that do **not** interchange:

| | what it is | where it comes from |
|---|---|---|
| **unconditional** | mean steps over *all* runs | the raw shards (Section 7) |
| **conditional** | mean steps over runs that *fixated* | `graph_statistics.csv` `mean_steps` |

`io.build_graph_statistics` aggregates `steps_success = when(fixation).then(steps)`, so the
published `mean_steps` is the **conditional** one. Since rho ~ 0.10 here, roughly 90% of runs go
extinct quickly and are excluded from it.

**Compute cost is driven by the unconditional time** - every run is paid for whether it fixates or
not. **The biology is the conditional one.** They need not share an exponent, so Section 9 fits
both and Section 12 costs from the unconditional fit only.

Because the C++ `step()` is O(1) (CSR adjacency, two-pool sampling, `moran_core.cpp:273`),
CPU time = unconditional steps / throughput. Throughput is measured in Section 11.

## What the prior batch does and does not tell you

`2026-06-10_scaling_study_6` covers N=10..100 at constant *density*, and its sparsest cell is a
spanning tree (`E = N-1`, k=2.0) fitting `T ~ N^2.281` on the *conditional* time.

That cell is a **different graph family**, not a baseline to reproduce: k=2.0 versus k=2.2. Section
10 measures how far apart they actually are, which is a result in its own right - if a 10% edge
surplus moves the answer a lot, this exponent will not transfer to the respiratory graphs unless
their mean degree matches too.

## Cost

A pilot (Section 2) puts the whole study at roughly **1.3 core-hours**, one batch, worst task under
4 minutes. An earlier draft budgeted 311 core-hours and split into three cost-tiered batches, on an
extrapolation from June's tree cell that turned out not to transfer. Two consequences:

- Tiering was dropped; one batch of 100 jobs is ample.
- **The pilot exponent drifts downward** (1.76 on N<=100, 1.31 on N>=100, both unconditional).
  If that survives into the real batch, extrapolating past N=1000 from a single exponent is not
  justified however good the global R^2 looks. The fix is to extend the ladder, which at this cost
  is nearly free. Section 9 tests it explicitly.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd /home/labs/pilpel/matanyaw/moran-process

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from moran_process.core.graph_zoo import GraphZoo
from moran_process.core.population_graph import PopulationGraph
from moran_process.pipeline.process_lab import ProcessLab

PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
SIM_DATA = PROJECT_ROOT / "simulation_data"
print(f"Project root: {PROJECT_ROOT}")

## Section 1 - Design constants

Everything that defines the experiment lives here. Nothing below this cell hard-codes a size,
a repeat count, or a job count.

In [ ]:
DATE = "2026_08_26"
DESCRIPTION = (
    "Scaling study: absorption time vs population size N, at constant mean degree "
    "k=2.2 (avian-like sparsity). Log-spaced ladder N=10..1000, r=1.1."
)
NOTES = (
    "Companion to 2026-06-10_scaling_study_6, which covers N=10..100 at constant "
    "density. NOT a replication: June's sparsest cell is a spanning tree (E=N-1, "
    "k~2.0); this zoo uses E=round(1.1N) (k=2.2). A pilot shows that 10% edge "
    "surplus cuts absorption time ~17x at N=100, so the two are different regimes."
)

# --- the ladder, log-spaced over two decades -------------------------------------
LADDER = [10, 15, 22, 32, 46, 68, 100, 150, 220, 320, 460, 680, 1000]

# One batch. A pilot (see Section 2) puts the whole study at ~1.3 core-hours with a
# worst single task under 4 min, so the cost-tiered split this notebook originally
# carried was solving a problem that does not exist at k=2.2. Tiering matters when
# per-simulation cost spans orders of magnitude across the ladder, because
# _create_task_list balances workers by SIM COUNT, not by cost. Here the spread is
# ~1100x but the absolute worst case is minutes, so it is not worth three batches.
# If you raise n_repeats or extend the ladder far past 1000, re-check that.
N_JOBS = 1000
MEMORY = "4GB"

# --- topology: constant mean degree, avian-like ----------------------------------
K_MEAN = 2.2  # avian_r4_l7 has N=31, E=34 -> k=2.194


def n_edges_for(n_nodes: int) -> int:
    """Edges at constant mean degree k, floored at a spanning tree."""
    return max(n_nodes - 1, round(K_MEAN * n_nodes / 2))


N_SEEDS = 200  # random graphs per rung
BASELINES = ("cycle", "star", "complete")  # one of each per rung

# --- simulation ------------------------------------------------------------------
R_VALUES = [1.1]
N_REPEATS = 10_000
SEED = 42  # batch_seed: simulation RNG
GRAPH_ZOO_SEED = 42  # random-graph topology RNG
ENGINE = "cpp"
QUEUE = "gsla-cpu"

# The engine default is 1e6. The pilot puts the largest MEAN at ~1.1e5 (N=1000), so
# the default would not censor the mean -- but absorption time is heavy-tailed and
# individual fixating runs run far above it, so some would clip. A censored run is
# written as (fixation=False, steps=max_steps), indistinguishable from extinction,
# and it biases mean_steps DOWN at exactly the large-N end being extrapolated from.
# 1e9 costs nothing here and removes the failure mode. Section 7 asserts nothing hit it.
MAX_STEPS = 1_000_000_000

# --- reference batch -------------------------------------------------------------
SIM_DATA = PROJECT_ROOT / "simulation_data"
JUNE_BATCH = SIM_DATA / "2026-06-10_scaling_study_6"

BATCH_NAME = f"{DATE}-size-study"
BATCH_DIR = SIM_DATA / BATCH_NAME

print(f"Ladder ({len(LADDER)} rungs): {LADDER}")
print(f"{'N':>6} {'E':>7} {'k':>6}")
for n in LADDER:
    e = n_edges_for(n)
    print(f"{n:>6} {e:>7} {2 * e / n:>6.2f}")
print(f"\nBatch: {BATCH_NAME}")
print(f"Graphs: {len(LADDER) * (N_SEEDS + len(BASELINES)):,}")

## Section 2 - Cost preview, before anything is built or submitted

The scaling study is itself subject to the scaling law, so price it first.

The numbers below are a **pilot measured directly at k=2.2** (5 graphs per rung, 400 repeats,
r=1.1) rather than extrapolated from June, because the June tree cell overestimates cost here by
~240x. They are a budgeting prior only: Section 9 re-fits the exponent from the real batch and
Section 11 re-measures throughput.

In [ ]:
# Pilot measured on a login node, 2026-08-23: 5 random graphs per rung at k=2.2,
# 400 repeats each, r=1.1, cpp engine. This is the budgeting prior ONLY; Section 9
# re-fits the exponent from the real batch and Section 11 re-measures throughput.
PILOT = pd.DataFrame(
    {
        "N": [10, 15, 22, 32, 46, 68, 100, 150, 220, 320, 460, 680, 1000],
        "T": [98, 242, 464, 936, 1491, 3734, 5674, 9812, 17188, 27989, 42427, 77618, 111222],
        "M_steps_per_sec": [77.0, 87.7, 93.7, 99.2, 103.0, 107.9, 110.8, 115.0,
                            120.5, 121.7, 123.9, 126.7, 129.5],
    }
).set_index("N")

_x, _y = np.log(PILOT.index.values.astype(float)), np.log(PILOT["T"].values)
PRIOR_ALPHA, _ic = np.polyfit(_x, _y, 1)
PRIOR_C = np.exp(_ic)
STEPS_PER_SEC = PILOT["M_steps_per_sec"].iloc[-1] * 1e6  # large-N rate; cost lives there

print(f"pilot prior: T = {PRIOR_C:.3f} * N^{PRIOR_ALPHA:.3f}")
print(f"pilot throughput: {PILOT.M_steps_per_sec.iloc[0]:.0f} M/s at N=10 "
      f"-> {PILOT.M_steps_per_sec.iloc[-1]:.0f} M/s at N=1000 (RISES with N)")
print(
    "\nNote: this does NOT match June's spanning-tree cell (alpha=2.281, T=97,560 at\n"
    "N=100). At k=2.2 the pilot gives 5,674 at N=100 -- 17x faster. A 10% edge surplus\n"
    "over a tree destroys the suppression, so the tree cell is not a proxy for avian.\n"
)


def prior_steps(n):
    return PRIOR_C * np.asarray(n, dtype=float) ** PRIOR_ALPHA


n_graphs_per_rung = N_SEEDS + len(BASELINES)
preview = pd.DataFrame(
    {
        "N": LADDER,
        "E": [n_edges_for(n) for n in LADDER],
        "graphs": n_graphs_per_rung,
        "steps_per_sim": [PILOT["T"].get(n, prior_steps(n)) for n in LADDER],
    }
)
preview["sec_per_graph"] = (
    preview.steps_per_sim * N_REPEATS * len(R_VALUES) / STEPS_PER_SEC
)
preview["core_hours"] = preview.sec_per_graph * preview.graphs / 3600

with pd.option_context("display.float_format", lambda v: f"{v:,.3f}"):
    print(preview.to_string(index=False))

n_sims = preview.graphs.sum() * N_REPEATS * len(R_VALUES)
# _create_task_list gives every worker an equal SIM COUNT. When that share is below
# N_REPEATS, expensive configs get split across workers; above it, one worker eats a
# whole config. Either way what matters is the wall time of the unluckiest worker.
share = n_sims / N_JOBS
worst = share * preview.steps_per_sim.max() / STEPS_PER_SEC
print(f"\nTOTAL: {preview.core_hours.sum():.2f} core-hours | {n_sims:,} simulations")
print(f"{N_JOBS} jobs | {share:,.0f} sims/worker | worst worker ~{worst / 60:.1f} min")
print(f"Longest single config (N={LADDER[-1]}): "
      f"{preview.steps_per_sim.max() * N_REPEATS / STEPS_PER_SEC:.0f} s")

over = preview[preview.steps_per_sim > 1e6].N.tolist()
print(f"\nRungs whose MEAN exceeds the 1e6 engine default: {over or 'none'}")
print(f"MAX_STEPS is set to {MAX_STEPS:,} (tail insurance, see Section 7)")

## Section 3 - Build the zoo

Random graphs at `E = round(1.1N)` plus one cycle, star and complete per rung.

Cycle and star are both k~2 and bracket the sparse regime: the cycle is the slow, neutral-ish
case, the star the extreme amplifier. Complete is the dense reference with known theory. They
give the topology-dependence of alpha for free, since they cost almost nothing to add.

In [ ]:
zoo = GraphZoo(name=BATCH_NAME)
for n in LADDER:
    if "cycle" in BASELINES:
        zoo.add(PopulationGraph.cycle_graph(n_nodes=n))
    if "star" in BASELINES:
        zoo.add(PopulationGraph.star_graph(n_nodes=n))
    if "complete" in BASELINES:
        zoo.add(PopulationGraph.complete_graph(n_nodes=n))
    e = n_edges_for(n)
    for seed in range(N_SEEDS):
        zoo.add(
            PopulationGraph.random_connected_graph(
                n_nodes=n, n_edges=e, seed=GRAPH_ZOO_SEED * 100_003 + seed
            )
        )

total_edges = sum(g.graph.number_of_edges() for g in zoo)
print(f"{len(zoo):,} graphs, {total_edges:,} edges")
# June's zoo.pkl came out at 54 bytes/edge; use that to predict pickle size.
print(f"~{total_edges * 54 / 1e6:.0f} MB pickled")
print(f"\nNote: the complete-graph baselines carry most of those edges "
      f"({sum(g.graph.number_of_edges() for g in zoo if g.category == 'Complete'):,}); "
      f"the k=2.2 study graphs are only "
      f"{sum(g.graph.number_of_edges() for g in zoo if g.category == 'Random'):,}.")

## Section 4 - Inspect before committing

Confirm the mean degree really is constant across the ladder. If `k` drifts, the fitted exponent
is measuring connectivity as well as N and the whole study is confounded.

In [ ]:
zoo_summary = pd.DataFrame(
    [
        dict(
            category=g.category,
            n_nodes=g.graph.number_of_nodes(),
            n_edges=g.graph.number_of_edges(),
            k=2 * g.graph.number_of_edges() / g.graph.number_of_nodes(),
        )
        for g in zoo
    ]
)

rnd = zoo_summary[zoo_summary.category == "Random"]
print("Random graphs, mean degree by rung (must be flat at k=2.2):")
print(rnd.groupby("n_nodes")["k"].agg(["mean", "min", "max", "count"]).round(3).to_string())

# If k drifts across the ladder, the fitted exponent measures connectivity as well as
# N and the study is confounded. Tolerance is loose at small N only because rounding
# E to an integer cannot hit 2.2 exactly there (N=15 -> E=16 -> k=2.13).
assert np.allclose(rnd.groupby("n_nodes")["k"].mean(), K_MEAN, atol=0.12), (
    "mean degree is not constant across the ladder"
)
print("\nGraphs by category:")
print(zoo_summary.groupby("category").size().to_string())

## Section 5 - Save and submit

**Requires `max_steps` to be plumbed through `submit_jobs` -> `task_manifest` -> `worker_lsf`.**
The engine constructor defaults to 1e6 and neither the pipeline nor the worker currently passes
anything else. At k=2.2 that default would not censor any rung's *mean* (largest is ~1.1e5 at
N=1000), but absorption time is heavy-tailed and individual fixating runs go far above the mean,
so some would clip. Section 7 checks.

Submission is in a separate cell from the save, and guarded, so a stray Run All does not fire a
job array.

In [ ]:
BATCH_DIR.mkdir(parents=True, exist_ok=True)
ZOO_PATH = BATCH_DIR / "zoo.pkl"
zoo.save(str(ZOO_PATH))
print(f"{len(zoo):,} graphs -> {ZOO_PATH}  ({ZOO_PATH.stat().st_size / 1e6:.1f} MB)")

In [ ]:
SUBMIT = True  # flip to True to actually submit

if not SUBMIT:
    print("SUBMIT is False - nothing submitted. Set SUBMIT = True to launch.")
else:
    zoo_on_disk = GraphZoo.load(str(ZOO_PATH))
    categories = sorted({g.category for g in zoo_on_disk})
    lab = ProcessLab()
    job_ids = lab.submit_jobs(
        zoo_path=str(ZOO_PATH),
        n_graphs=len(zoo_on_disk),
        r_values=R_VALUES,
        batch_name=BATCH_NAME,
        batch_dir=str(BATCH_DIR),
        n_repeats=N_REPEATS,
        n_requested_jobs=N_JOBS,
        queue=QUEUE,
        memory=MEMORY,
        graph_types=categories,
        node_sizes=LADDER,
        description=DESCRIPTION,
        notes=NOTES,
        batch_seed=SEED,
        engine=ENGINE,
        max_steps=MAX_STEPS,  # <-- needs the plumbing change described above
        zoo_config=dict(
            graph_zoo_seed=GRAPH_ZOO_SEED,
            k_mean=K_MEAN,
            n_seeds=N_SEEDS,
            baselines=list(BASELINES),
            sizes=LADDER,
            edge_rule="E = max(N-1, round(K_MEAN*N/2))",
        ),
    )
    print(job_ids)

---
# Analysis

Everything below runs **after** the three batches have finished and their aggregation jobs have
written `graph_statistics.csv`. It is file reads and cheap fitting only, per the builder/reader
split: no shard scanning, no simulation.

In [ ]:
from moran_process.pipeline.post_batch import post_batch_status

post_batch_status(str(BATCH_DIR), r_values=R_VALUES)

## Section 7 - Raw shard pass: censoring, and the two different "times"

**Run this before looking at any fit.** It does one pass over the raw shards and returns three
things per graph.

### Censoring

`worker_lsf` now writes an explicit `censored` column (True when the run was stopped by
`max_steps` rather than absorbing). Before that column existed, a truncated run was written as
`(fixation=False, steps=max_steps)` and was indistinguishable from a fast extinction, which pulls
`mean_steps` *down* and flattens the curve at exactly the large-N end you extrapolate from.

### Two times, and why `graph_statistics.csv` only has one

`io.build_graph_statistics` aggregates `steps_success = when(fixation).then(steps)`, so
**`mean_steps` in `graph_statistics.csv` is conditional on fixation.** That is the biological
quantity: how long a successful invasion takes.

It is *not* the compute-cost driver. Every run is paid for, including the ~90% that go extinct
quickly, so cost is set by the **unconditional** mean over all runs. The two differ by a large
factor here (rho ~ 0.10, and extinctions are much shorter than fixations), and they need not even
share an exponent.

So this pass computes both, and Section 9 fits them separately:

| quantity | source | answers |
|---|---|---|
| `T_cond` | `mean_steps` (conditional) | how long fixation takes - the biology |
| `T_uncond` | this shard pass | what the batch costs - the planning question |

In [ ]:
import polars as pl

shards = BATCH_DIR / "tmp" / "results" / "*.parquet"
lf = pl.scan_parquet(str(shards))
names = lf.collect_schema().names()

# `censored` is written by worker_lsf; batches from before that change lack it, in
# which case fall back to the (exact, same-semantics) steps == max_steps test.
censored_expr = (
    pl.col("censored") if "censored" in names else (pl.col("steps") >= MAX_STEPS)
)

raw = (
    lf.group_by("wl_hash")
    .agg(
        pl.len().alias("n_runs"),
        censored_expr.sum().alias("n_censored"),
        pl.col("steps").max().alias("max_steps_seen"),
        # Cost driver: every run is paid for, fixating or not.
        pl.col("steps").mean().alias("T_uncond"),
        # Biology: matches graph_statistics.mean_steps.
        pl.when(pl.col("fixation")).then(pl.col("steps")).mean().alias("T_cond"),
        pl.col("fixation").mean().alias("rho"),
        pl.col("duration").sum().alias("sim_sec"),
    )
    .collect()
    .to_pandas()
)
print(f"{'(new censored column)' if 'censored' in names else '(derived from steps)'}")
print(f"Runs at the ceiling: {raw.n_censored.sum():,} / {raw.n_runs.sum():,}")
print(f"Largest step count observed: {raw.max_steps_seen.max():,} of {MAX_STEPS:,}")

assert raw.n_censored.sum() == 0, (
    "CENSORED RUNS PRESENT - both T_cond and T_uncond are biased low at the affected "
    "graphs. Raise MAX_STEPS and re-run before fitting."
)
print("No censoring. Both fits below are unbiased.\n")

# Attach graph size. graph_props.csv is the join partner for wl_hash.
props = pd.read_csv(BATCH_DIR / "graph_props.csv")
raw = raw.merge(props[["wl_hash", "category", "n_nodes", "n_edges"]], on="wl_hash")

print("The two times are not interchangeable:")
chk = raw[raw.category == "Random"].groupby("n_nodes")[["T_uncond", "T_cond", "rho"]].mean()
chk["ratio"] = chk.T_cond / chk.T_uncond
print(chk.round(2).to_string())

## Section 8 - Load results

Reads `graph_statistics.csv` through the documented reader, which raises rather than silently
starting a shard scan inside the kernel.

The cycle / star / complete baselines are simulated alongside but are kept out of the main fit:
June shows alpha ranging 1.83 (complete) to 2.70 (cycle, star), so the exponent is a property of
the topology family and pooling them would average distinct laws into a meaningless middle.

In [ ]:
from moran_process.analysis.analysis_utils.io import load_graph_statistics

# graph_statistics.csv is the documented reader path, but note its mean_steps is
# CONDITIONAL on fixation (io.py builds steps_success = when(fixation).then(steps)).
# Section 7's shard pass already gives both times, so this is mainly a cross-check
# that the two routes agree on the conditional number.
stats = load_graph_statistics(str(BATCH_DIR), r_filter=R_VALUES)
print(f"{len(stats):,} (graph, r) rows")
print(stats.groupby("category").size().to_string())

xchk = stats[["wl_hash", "mean_steps"]].merge(raw[["wl_hash", "T_cond"]], on="wl_hash")
rel = ((xchk.mean_steps - xchk.T_cond).abs() / xchk.T_cond).max()
print(f"\nmean_steps vs shard-pass T_cond: max relative difference {rel:.2e}")
assert rel < 1e-6, "graph_statistics.mean_steps disagrees with the shard pass"

# The sparse k=2.2 random graphs are the study's main series. The cycle/star/complete
# baselines are simulated too but are NOT pooled: June shows alpha ranging 1.83 to
# 2.70 across topologies, so pooling would average distinct laws into a meaningless
# middle. Section 10 fits them separately.
sparse = raw[raw.category == "Random"]


def series(col):
    g = sparse.groupby("n_nodes")[col].agg(["mean", "std", "count"])
    g.columns = ["T", "sd", "n_graphs"]
    g["sem"] = g.sd / np.sqrt(g.n_graphs)
    return g


curve_cost = series("T_uncond")  # what the batch costs
curve_bio = series("T_cond")  # how long fixation takes

print("\nUNCONDITIONAL (cost driver):")
print(curve_cost.round(1).to_string())
print("\nCONDITIONAL on fixation (biology):")
print(curve_bio.round(1).to_string())

# The pilot in Section 2 measured mean(steps) over ALL runs, so it is comparable to
# curve_cost only. Comparing it to the conditional series would be apples to oranges.
print("\nMeasured (unconditional) vs pilot prior:")
for n in curve_cost.index:
    if n in PILOT.index:
        m, p = curve_cost.loc[n, "T"], PILOT.loc[n, "T"]
        print(f"  N={n:>5}: measured {m:>10,.0f}  pilot {p:>10,.0f}  ratio {m / p:.2f}")

## Section 9 - Linear, polynomial, or exponential?

Three nested claims, discriminated on the same data:

| model | linearised form | signature |
|---|---|---|
| linear | `T ~ N` | power law with `alpha = 1` |
| polynomial | `log T ~ alpha * log N` | straight on log-log |
| exponential | `log T ~ beta * N` | straight on lin-log |

R^2 alone is a weak discriminator over a short range, so this also reports AIC (same response
`log T`, same parameter count, so AIC is directly comparable) and a **curvature test**: fit the
exponent on the bottom half and the top half of the ladder separately. A true power law gives the
same `alpha` in both; a drifting `alpha` means the single-exponent extrapolation is unsafe no
matter how good the global R^2 looks.

In [ ]:
def fit_line(x, y):
    """OLS slope/intercept plus R^2, AIC and the slope's standard error."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    n = len(x)
    slope, intercept = np.polyfit(x, y, 1)
    resid = y - (slope * x + intercept)
    rss = float(resid @ resid)
    r2 = 1 - rss / float(((y - y.mean()) ** 2).sum())
    aic = n * np.log(rss / n) + 2 * 3  # slope, intercept, sigma
    se = np.sqrt(rss / (n - 2) / ((x - x.mean()) ** 2).sum())
    return dict(slope=slope, intercept=intercept, r2=r2, aic=aic, se=se, n=n)


def analyse(curve, label):
    """Discriminate power law vs exponential, and test whether alpha is stable."""
    N = curve.index.values.astype(float)
    logT = np.log(curve["T"].values)
    power, expo = fit_line(np.log(N), logT), fit_line(N, logT)

    a, se = power["slope"], power["se"]
    lo, hi = a - 1.96 * se, a + 1.96 * se
    d_aic = expo["aic"] - power["aic"]

    print(f"=== {label} ===")
    print(f"  power law   T ~ N^{a:.3f} +/- {1.96 * se:.3f}"
          f"   R2={power['r2']:.5f}  AIC={power['aic']:.1f}")
    print(f"  exponential T ~ exp({expo['slope']:.5f} N)"
          f"        R2={expo['r2']:.5f}  AIC={expo['aic']:.1f}")
    print(f"  delta AIC = {abs(d_aic):.1f} favouring "
          f"{'POWER LAW' if d_aic > 0 else 'EXPONENTIAL'}")
    print(f"  alpha = {a:.3f}  (95% CI {lo:.3f} .. {hi:.3f})")
    for name, val in [("linear", 1), ("quadratic", 2), ("cubic", 3)]:
        verdict = "consistent" if lo <= val <= hi else "ruled out"
        print(f"    {name} (alpha={val})? {verdict}")

    # Curvature: a true power law gives the same exponent on both halves. A drifting
    # alpha means the single-exponent extrapolation is unsafe however good R^2 is.
    h = len(N) // 2
    f_lo = fit_line(np.log(N[: h + 1]), logT[: h + 1])
    f_hi = fit_line(np.log(N[h:]), logT[h:])
    drift = abs(f_hi["slope"] - f_lo["slope"])
    pooled = np.hypot(f_lo["se"], f_hi["se"])
    print(f"  curvature: alpha(N<={N[h]:.0f})={f_lo['slope']:.3f}"
          f"  alpha(N>={N[h]:.0f})={f_hi['slope']:.3f}"
          f"  drift={drift:.3f} ({drift / pooled:.1f} sigma)")
    print("  -> " + ("stable; single exponent extrapolates"
                     if drift < 2 * pooled
                     else "DRIFTING; do not extrapolate from one exponent"))
    print()
    return dict(power=power, expo=expo, alpha=a, lo=lo, hi=hi, N=N, logT=logT)


fit_cost = analyse(curve_cost, "UNCONDITIONAL (drives compute cost)")
fit_bio = analyse(curve_bio, "CONDITIONAL on fixation (biology)")

if abs(fit_cost["alpha"] - fit_bio["alpha"]) > 1.96 * np.hypot(
    fit_cost["power"]["se"], fit_bio["power"]["se"]
):
    print("NOTE: the two exponents differ significantly. Cost and biology scale "
          "differently here, so neither number answers both questions.")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
SERIES = [
    (curve_cost, fit_cost, "unconditional (cost)", "steelblue"),
    (curve_bio, fit_bio, "conditional (biology)", "darkorange"),
]

# log-log: a power law is a straight line here
ax = axes[0]
for curve, f, label, c in SERIES:
    ax.errorbar(f["N"], curve["T"], yerr=curve["sem"], fmt="o", color=c,
                label=f"{label}: $N^{{{f['alpha']:.2f}}}$")
    grid = np.logspace(np.log10(f["N"].min()), np.log10(f["N"].max()), 100)
    ax.plot(grid, np.exp(f["power"]["intercept"]) * grid ** f["alpha"], "-", color=c, lw=1)
ax.set(xscale="log", yscale="log", xlabel="N (nodes)", ylabel="mean steps",
       title="log-log: straight = polynomial")
ax.legend(fontsize=8)

# lin-log: an exponential is a straight line here
ax = axes[1]
for curve, f, label, c in SERIES:
    ax.errorbar(f["N"], curve["T"], yerr=curve["sem"], fmt="o", color=c, label=label)
    ax.plot(f["N"], np.exp(f["expo"]["intercept"] + f["expo"]["slope"] * f["N"]),
            "-", color=c, lw=1)
ax.set(yscale="log", xlabel="N (nodes)", ylabel="mean steps",
       title="lin-log: straight = exponential")
ax.legend(fontsize=8)

# local exponent: flat means a genuine single-exponent power law
ax = axes[2]
for curve, f, label, c in SERIES:
    local = np.diff(f["logT"]) / np.diff(np.log(f["N"]))
    mid = np.sqrt(f["N"][:-1] * f["N"][1:])
    ax.plot(mid, local, "o-", color=c, label=label)
    ax.axhline(f["alpha"], ls="--", color=c, lw=1)
ax.set(xscale="log", xlabel="N (geometric midpoint)", ylabel="local exponent",
       title="local $d\\log T/d\\log N$: flat = true power law")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## Section 10 - How sharply does sparsity matter?

June's minimum-density cell (`E = N-1`, k=2.0) versus this study's `E = 1.1N` (k=2.2). Both are
constant-mean-degree families differing by a 10% edge surplus, so the comparison isolates how
sensitive the Moran process is right at the spanning-tree boundary.

This is **not** a validation of the pipeline, because the two are genuinely different graph
families and are expected to differ. It is a measurement in its own right, and a warning: if a
10% edge change moves the answer a lot, then the exponent fitted here does not transfer to the
respiratory graphs unless their mean degree also matches.

Both series here are **conditional** fixation time, since that is what June's
`graph_statistics.csv` stores.

In [ ]:
# June's graph_statistics.csv also carries the CONDITIONAL mean_steps, so it must be
# compared against curve_bio, not curve_cost. June's raw shards would be needed for an
# unconditional comparison and are not read here.
june = pd.read_csv(JUNE_BATCH / "graph_statistics.csv")
june = june[(june.category == "Random") & (june.n_edges == june.n_nodes - 1)]
june_curve = june.groupby("n_nodes")["mean_steps"].mean()
june_fit = fit_line(np.log(june_curve.index.values.astype(float)), np.log(june_curve.values))

print("Conditional fixation time, both constant-degree families:")
print(f"  this study (k=2.2): alpha = {fit_bio['alpha']:.3f}")
print(f"  June       (k=2.0): alpha = {june_fit['slope']:.3f}")
print(f"  difference        : {abs(fit_bio['alpha'] - june_fit['slope']):.3f}")

shared = [n for n in curve_bio.index if n in june_curve.index]
if shared:
    print("\nRatio at shared sizes (k=2.0 / k=2.2), i.e. the cost of near-tree sparsity:")
    for n in shared:
        print(f"  N={n:>5}: June {june_curve[n]:>10,.0f}  this {curve_bio.loc[n, 'T']:>10,.0f}"
              f"  ratio {june_curve[n] / curve_bio.loc[n, 'T']:>6.1f}x")

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(june_curve.index, june_curve.values, "s--", color="gray",
        label=f"June, E=N-1, k=2.0 ($\\alpha$={june_fit['slope']:.2f})")
ax.plot(curve_bio.index, curve_bio["T"], "o-", color="darkorange",
        label=f"this study, E=1.1N, k=2.2 ($\\alpha$={fit_bio['alpha']:.2f})")
ax.set(xscale="log", yscale="log", xlabel="N",
       ylabel="conditional fixation time (steps)",
       title="Sensitivity to sparsity at the tree boundary")
ax.legend()
plt.tight_layout()
plt.show()

## Section 11 - Throughput: is a step really O(1)?

`step()` is O(1) in work, but not necessarily in *memory*: `state_`, `order_` and `loc_` all grow
with N and are accessed randomly, so throughput could fall at large N and a pure steps
extrapolation would then understate cost.

The pilot says it does **not** fall — it *rises*, 77 M steps/s at N=10 to 129 M at N=1000. At
k=2.2 the CSR array stays tiny (1100 edges at N=1000), so cache is never the binding constraint;
what dominates at small N is per-repeat setup, which is amortised over only ~98 steps at N=10 but
over ~111,000 at N=1000. So the O(1) assumption is safe here, and if anything conservative.

This cell re-measures it on whatever machine you are on.

**Run this in an `inode` session, not on the login node.**

In [ ]:
import time
from moran_process.simulations.cpp_moran_wrapper import CppMoranProcess

RUN_THROUGHPUT_CALIBRATION = False  # flip on inside an inode session

if not RUN_THROUGHPUT_CALIBRATION:
    print("Skipped. Set RUN_THROUGHPUT_CALIBRATION = True inside an inode session.")
else:
    rows = []
    for n in LADDER:
        g = PopulationGraph.random_connected_graph(
            n_nodes=n, n_edges=n_edges_for(n), seed=0
        )
        # Keep total work roughly constant per rung so every point costs the same.
        reps = max(20, int(2e7 / max(prior_steps(n), 1)))
        sim = CppMoranProcess(
            graph_core=g.to_simulation_struct(),
            selection_coefficient=R_VALUES[0],
            max_steps=MAX_STEPS,
            seed=0,
        )
        t0 = time.perf_counter()
        out = sim.run_repeats(reps)
        wall = time.perf_counter() - t0
        steps = int(np.sum(out["steps"]))
        rows.append(
            dict(N=n, reps=reps, steps=steps, wall_sec=wall, steps_per_sec=steps / wall)
        )
        print(f"  N={n:>5}  {reps:>7} reps  {steps / wall / 1e6:>7.1f} M steps/s")

    thru = pd.DataFrame(rows).set_index("N")
    falloff = thru.steps_per_sec.iloc[0] / thru.steps_per_sec.iloc[-1]
    print(f"\nThroughput falls {falloff:.2f}x from N={LADDER[0]} to N={LADDER[-1]}")
    print(
        "O(1) holds in practice" if falloff < 1.5
        else "NOT flat - the seconds extrapolation must carry this factor"
    )
    STEPS_PER_SEC_MEASURED = float(thru.steps_per_sec.iloc[-1])  # conservative: large-N rate

## Section 12 - What would a bigger experiment cost?

The payoff. Combines the fitted exponent (with its CI, so the answer is a band and not a
false-precision point) with the measured large-N throughput.

Read the band, not the centre: at N=10000 a +/-0.1 uncertainty in `alpha` is a factor of
`10000^0.1 = 2.5` in cost.

In [ ]:
# Cost is driven by the UNCONDITIONAL time: every run is paid for, fixating or not.
# Using the conditional exponent here would be a category error.
f = fit_cost
rate = globals().get("STEPS_PER_SEC_MEASURED", STEPS_PER_SEC)
print(f"Using {rate / 1e6:.1f} M steps/s/core"
      f"{' (measured)' if 'STEPS_PER_SEC_MEASURED' in globals() else ' (pilot - run Section 11)'}")
print(f"Using the UNCONDITIONAL exponent alpha = {f['alpha']:.3f}\n")

TARGETS = [1000, 1500, 2000, 3000, 5000, 10000]
PLANNED_GRAPHS = N_SEEDS + len(BASELINES)


def cost_core_h(n, a):
    steps = np.exp(f["power"]["intercept"]) * float(n) ** a
    return steps, steps * N_REPEATS * len(R_VALUES) / rate * PLANNED_GRAPHS / 3600


extrap = pd.DataFrame(
    [
        dict(
            N=n,
            steps_per_sim=cost_core_h(n, f["alpha"])[0],
            core_h=cost_core_h(n, f["alpha"])[1],
            core_h_lo=cost_core_h(n, f["lo"])[1],
            core_h_hi=cost_core_h(n, f["hi"])[1],
        )
        for n in TARGETS
    ]
)

print(f"Cost of one rung of {PLANNED_GRAPHS} graphs at {N_REPEATS:,} repeats, r={R_VALUES}:")
print(f"{'N':>7} {'steps/sim':>13} {'core-hours':>12}   {'95% band':>22}")
for r in extrap.itertuples():
    print(f"{r.N:>7} {r.steps_per_sim:>13,.0f} {r.core_h:>12,.2f}"
          f"   [{r.core_h_lo:>9,.2f} .. {r.core_h_hi:<9,.2f}]")

print(f"\nThis study cost ~{preview.core_hours.sum():.2f} core-hours (predicted).")
biggest = extrap.steps_per_sim.max()
print(f"MAX_STEPS headroom at N={TARGETS[-1]}: mean uses {biggest / MAX_STEPS:.2e} of the cap.")
print("Note the band widens fast: a +/-0.1 uncertainty in alpha is a factor of "
      f"{10000 ** 0.1:.1f} at N=10000. Read the band, not the centre.")

fig, ax = plt.subplots(figsize=(7, 5))
ax.fill_between(extrap.N, extrap.core_h_lo, extrap.core_h_hi, alpha=0.25,
                color="crimson", label="95% band from $\\alpha$ CI")
ax.plot(extrap.N, extrap.core_h, "o-", color="crimson", label="predicted")
ax.axhline(preview.core_hours.sum(), ls=":", color="gray", label="this study")
ax.set(xscale="log", yscale="log", xlabel="N (nodes)",
       ylabel=f"core-hours per rung of {PLANNED_GRAPHS} graphs",
       title="Extrapolated cost (unconditional time)")
ax.legend()
plt.tight_layout()
plt.show()

## Section 13 - Conclusion

Fill in from the numbers above. Note there are **two** answers, not one:

| | alpha | 95% CI | delta AIC vs exp | alpha stable? |
|---|---|---|---|---|
| unconditional (cost) | ____ | ____ | ____ | ____ |
| conditional (biology) | ____ | ____ | ____ | ____ |

- **Linear ruled out?** ____
- **Practical ceiling:** at ____ core-hours per rung, N = ____ is the largest size worth running
  at these repeat counts.

The pilot's unconditional answer, for reference, was **polynomial, alpha ~ 1.5**, R^2 = 0.992
power against 0.678 exponential. Exponential is not close, and linear is ruled out too.

### Caveats to carry forward

1. **The exponent drifts.** The pilot gives 1.76 on N<=100 and 1.31 on N>=100. This is the single
   most important thing to check in Section 9. If it holds, `T ~ N^alpha` with one alpha is a
   local description rather than a law, and extrapolating to N=10000 is unjustified regardless of
   global R^2. The fix is to extend the ladder, not to trust the fit. At ~1.3 core-hours for
   10..1000, reaching N=5000 costs little more.
2. **Cost and biology may not share an exponent.** Section 9 tests this directly. If they differ,
   quoting one number as "the" scaling of this simulation is wrong, and which one you want depends
   on whether you are planning compute or describing evolution.
3. **One selection coefficient.** Everything is r=1.1. Absorption time behaves very differently
   near r=1 (diffusive rather than driven), so cost estimates for an r-sweep that includes
   near-neutral values will be understated.
4. **One sparsity, and sparsity matters sharply.** k=2.2 only. Section 10 quantifies the gap to
   k=2.0. Do not reuse this alpha at another density, and note that the respiratory graphs sit in
   exactly this sensitive near-tree region.
5. **The pilot in Section 2 was measured on a login node** on 2026-08-23, at 5 graphs x 400 repeats
   per rung. It is a budgeting prior only; every number it feeds is recomputed from the real batch
   in Sections 7-12.